In [ ]:
!pip install duckdb # package to run SQL queries on dataframes

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import duckdb
import requests
import numpy as np
import os
import time
from google.colab import files

In [ ]:
API_KEY = "709be85ec89c41b6ee2627e05e0697f325f0456a"
STATE_FIPS = '36'
NYC_COUNTY_FIPS = ['005', '047', '061', '081', '085']

# 2. Define Census Variables
CENSUS_VARIABLES = {

    # Total Population - for whom poverty status is determined
    'S1701_C01_001E':'S1701_C01_001E',

    # Population below the poverty line
    'S1701_C02_001E':'S1701_C02_001E',

    # Households
    'S2201_C01_001E':'S2201_C01_001E', # Total households

    'S1901_C01_012E':'S1901_C01_012E', # Median Household Income (dollars)
    'S1901_C01_013E':'S1901_C01_013E', # Mean Household Income (dollars)

    'S2201_C03_001E':'S2201_C03_001E', # Households receiving SNAP
    'S2201_C03_021E':'S2201_C03_021E', # Households receiving SNAP + Below poverty level in the past 12 months
    'S2201_C03_034E':'S2201_C03_034E', # Households receiving SNAP + Median HOUSEHOLD INCOME IN THE PAST 12 MONTHS
    'S2201_C04_001E':'S2201_C04_001E', # Percent households receiving SNAP
    'S2201_C04_021E':'S2201_C04_021E', # Percent households receiving SNAP + Below poverty level in the past 12 months

    }

VARIABLE_CODES = list(CENSUS_VARIABLES.keys())

In [ ]:
# Data Fetching Function
def fetch_census_data(api_key, state_fips, county_fips_list, variables_dict):
    all_data = []
    variable_string = ','.join(variables_dict.keys()) + ',NAME'

    print(f"\nFetching data for {len(variables_dict)} variables...")

    for county_fips in county_fips_list:
        print(f"Fetching data for County FIPS {county_fips}...")
        url = (
            f"https://api.census.gov/data/2023/acs/acs5"
            f"get={variable_string}&"
            f"for=tract:*&"
            f"in=state:{state_fips}&"
            f"in=county:{county_fips}&"
            f"key={api_key}"
        )

        try:
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
            headers = data[0]
            rows = data[1:]
            df = pd.DataFrame(rows, columns=headers)
            all_data.append(df)
            time.sleep(0.5)

        except Exception as e:
            print(f"Error for county {county_fips}: {e}. Skipping.")

    if not all_data:
        return pd.DataFrame()

    final_df = pd.concat(all_data, ignore_index=True)
    final_df['GEOID'] = final_df['state'] + final_df['county'] + final_df['tract']

    final_columns = ['GEOID', 'NAME'] + VARIABLE_CODES
    final_df = final_df[final_columns].copy()

    return final_df

In [ ]:
# Execution & Processing

census_tract_data = fetch_census_data(API_KEY, STATE_FIPS, NYC_COUNTY_FIPS, CENSUS_VARIABLES)

# Convert relevant columns to numeric, coercing errors to NaN
for col in VARIABLE_CODES:
    census_tract_data[col] = pd.to_numeric(census_tract_data[col], errors='coerce')

# Now, explicitly replace -666666666 with NaN
census_tract_data.replace(-666666666, np.nan, inplace=True)

display(census_tract_data.head())


Fetching data for 10 variables...
Fetching data for County FIPS 005...
Fetching data for County FIPS 047...
Fetching data for County FIPS 061...
Fetching data for County FIPS 081...
Fetching data for County FIPS 085...


,GEOID,NAME,S1701_C01_001E,S1701_C02_001E,S2201_C01_001E,S1901_C01_012E,S1901_C01_013E,S2201_C03_001E,S2201_C03_021E,S2201_C03_034E,S2201_C04_001E,S2201_C04_021E
0,36005000100,Census Tract 1; Bronx County; New York,0,0,0,NaN,NaN,0,0,NaN,NaN,NaN
1,36005000200,Census Tract 2; Bronx County; New York,5177,729,1446,121171.0,130645.0,240,21,119167.0,16.6,8.8
2,36005000400,Census Tract 4; Bronx County; New York,6481,317,2246,98242.0,113925.0,335,48,155665.0,14.9,14.3
3,36005001600,Census Tract 16; Bronx County; New York,5759,1044,2149,42957.0,61302.0,947,575,20216.0,44.1,60.7
4,36005001901,Census Tract 19.01; Bronx County; New York,2396,730,1092,67361.0,89340.0,265,159,14438.0,24.3,60.0


In [ ]:
new_column_names = {

    'S1701_C01_001E': 'Total Population',
    'S1701_C02_001E': 'Population below poverty line',
    'S2201_C01_001E': 'Total Households',
    'S1901_C01_012E': 'Median HH Income (dollars)',
    'S1901_C01_013E': 'Mean HH Income (dollars)',
    'S2201_C03_001E': 'HH receiving SNAP',
    'S2201_C03_021E': 'HH w/SNAP + Below poverty lvl, past 12 months',
    'S2201_C03_034E': 'HH w/SNAP, Median HH Income past 12 months',
    'S2201_C04_001E': 'Percent HH w/SNAP',
    'S2201_C04_021E': 'Percent HH w/SNAP + Below poverty lvl, past 12 months'

}

census_tract_data2 = census_tract_data.rename(columns=new_column_names)

In [ ]:
census_tract_data2.head()

,GEOID,NAME,Total Population,Population below poverty line,Total Households,Median HH Income (dollars),Mean HH Income (dollars),HH receiving SNAP,"HH w/SNAP + Below poverty lvl, past 12 months","HH w/SNAP, Median HH Income past 12 months",Percent HH w/SNAP,"Percent HH w/SNAP + Below poverty lvl, past 12 months"
0,36005000100,Census Tract 1; Bronx County; New York,0,0,0,NaN,NaN,0,0,NaN,NaN,NaN
1,36005000200,Census Tract 2; Bronx County; New York,5177,729,1446,121171.0,130645.0,240,21,119167.0,16.6,8.8
2,36005000400,Census Tract 4; Bronx County; New York,6481,317,2246,98242.0,113925.0,335,48,155665.0,14.9,14.3
3,36005001600,Census Tract 16; Bronx County; New York,5759,1044,2149,42957.0,61302.0,947,575,20216.0,44.1,60.7
4,36005001901,Census Tract 19.01; Bronx County; New York,2396,730,1092,67361.0,89340.0,265,159,14438.0,24.3,60.0


In [ ]:
census_tract_data2.to_csv('NYC_CensusData_Poverty & SNAP.csv', index=False)
print(f"\nSUCCESS: Data saved!")
files.download('NYC_CensusData_Poverty & SNAP.csv')


SUCCESS: Data saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>